In [10]:
import math
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple


rho_w = 1025.0       # [kg/m^3] Density of seawater
g = 9.81             # [m/s^2] Gravitational acceleration
Hs = 0.7            # [m] Significant wave height (Hm0)
sop = 0.04            # [s] Peak wave period
tan_beta = 1.0 / 3.0 # [-] Slope of the revetment (tan(beta) = 1/3 for 1:3 slope)
Tp = math.sqrt((Hs * 2 * math.pi) / (g * sop))          # [s] Peak period of the wave (T_p)
# --- Filter and Soil Properties ---

D = 0.15             # [m] Thickness of the top (impervious) layer
b = 0.10             # [m] Thickness of the filter layer beneath --reset=0.13
k = 0.3             # [m/s] Permeability of the filter layer
k_prime = 0.01       # [m/s] Permeability of the base layer below the filter

# --- Derived Parameter ---

Lambda = np.sqrt(D * b * k / k_prime)  # [m] Leakage length (controls pressure dissipation)

#----------------------------------------------------------------------------------------------------------------------------

# Step 1: Estimate mean period
Tm_1_0 = Tp / 1.1

# Step 2: Calculate wavelength
L_m_1_0 = (g * Tm_1_0**2) / (2.0 * np.pi)

# Step 3: Calculate Iribarren number
xi_m_1_0 = tan_beta / np.sqrt(Hs / L_m_1_0)
print(f"Iribarren number: {xi_m_1_0:.3f}")
print(Tp)


Iribarren number: 1.515
3.347915243953695


In [2]:
from docx import Document
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple

# Load the document
doc = Document("data.docx")

# Get the first table
table = doc.tables[0]

# Extract the table into a list of lists
data = []
for row in table.rows:
    data.append([cell.text.strip() for cell in row.cells])

# Convert to DataFrame
df = pd.DataFrame(data[1:], columns=data[0])

# Clean up column names (strip spaces, replace weird characters)
df.columns = [col.strip().replace("", "Δ") for col in df.columns]

# Check exact column names
print("Columns:", df.columns.tolist())

# Filter rows where Type == "CB" and Toplayer == "Rectangular blocks"
df_cb_rect = df[
    (df["Type"] == "CB") &
    (df["Toplayer"] == "Rectangular blocks")
]

# Convert relevant columns to numeric
df_cb_rect["Hs/ΔD"] = pd.to_numeric(df_cb_rect["Hs/ΔD"], errors="coerce")
df_cb_rect["N"] = pd.to_numeric(df_cb_rect["N"], errors="coerce")

# Drop rows with missing numeric data
df_cb_rect = df_cb_rect.dropna(subset=["Hs/ΔD", "N", "Updated damage"])

# Define color mapping
damage_colors = {
    "0": "white",
    "a": "green",
    "b": "yellow",
    "c": "orange",
    "d": "red",
    "c1": "orange",
    "d1": "red",
}

# Split into circles and crosses
df_circles = df_cb_rect[~df_cb_rect["Updated damage"].isin(["c1", "d1"])]
df_crosses = df_cb_rect[df_cb_rect["Updated damage"].isin(["c1", "d1"])]

# --- Add your own data point ---
my_N = 1000
my_damage = "d"

# Define densities
rho_s = 2300     # block density (kg/m³)
rho_w = 1000     # water density (kg/m³)

# Calculate Δ
Delta = (rho_s - rho_w) / rho_w

# Calculate Hs/(Δ·D)
Hs = 0.7
D = 0.15
my_Hs_Delta_D = Hs / (Delta * D)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

# Circles
ax.scatter(
    df_circles["N"],
    df_circles["Hs/ΔD"],
    c=df_circles["Updated damage"].map(damage_colors),
    edgecolor='black',
    s=80,
    marker='o'
)

# Crosses
ax.scatter(
    df_crosses["N"],
    df_crosses["Hs/ΔD"],
    c=df_crosses["Updated damage"].map(damage_colors),
    edgecolor='black',
    s=80,
    marker='+',
    linewidth=2
)

# Plot your own data point
ax.scatter(
    my_N,
    my_Hs_Delta_D,
    color=damage_colors[my_damage],
    edgecolor='black',
    s=80,
    marker='s',
    linewidth=1.5
)

# Labels and title
ax.set_xlim(0, 2000)
ax.set_xlabel("N", fontsize=14)
ax.set_ylabel("Hs/ΔD", fontsize=14)
ax.set_title("Hs/ΔD vs N for Type CB with Rectangular blocks", fontsize=16)

# Remove legend entirely (no ax.legend call)

# Ticks formatting
ax.tick_params(axis='x', labelrotation=45, labelsize=12)
ax.tick_params(axis='y', labelsize=12)

# Grid
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Optional: print filtered dataframe
print(df_cb_rect)


ModuleNotFoundError: No module named 'docx'